In [1]:
"""
NCAA March Madness 2026 Predictions using Neural Network
Predicts win probabilities for all possible matchups in 2026 tournament
"""

# Simple neural network implementation without external libraries
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

In [ ]:
warnings.filterwarnings("ignore")

# Data paths
DATA_DIR = Path("../data")

def load_data():
    """Load all necessary data files"""
    print("Loading data files...")

    # Teams
    m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
    w_teams = pd.read_csv(DATA_DIR / "WTeams.csv")

    # Seeds
    m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
    w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

    # Tournament results (compact)
    m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
    w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

    # Detailed results (regular season and tournament)
    m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
    w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
    m_tourney_detailed = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
    w_tourney_detailed = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")

    # Rankings (men only)
    m_rankings = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")

    # Conferences
    m_conf = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
    w_conf = pd.read_csv(DATA_DIR / "WTeamConferences.csv")

    # Sample submission
    sample_sub = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

    return {
        "m_teams": m_teams,
        "w_teams": w_teams,
        "m_seeds": m_seeds,
        "w_seeds": w_seeds,
        "m_tourney_compact": m_tourney_compact,
        "w_tourney_compact": w_tourney_compact,
        "m_reg_detailed": m_reg_detailed,
        "w_reg_detailed": w_reg_detailed,
        "m_tourney_detailed": m_tourney_detailed,
        "w_tourney_detailed": w_tourney_detailed,
        "m_rankings": m_rankings,
        "m_conf": m_conf,
        "w_conf": w_conf,
        "sample_sub": sample_sub,
    }


def add_gender(data_dict):
    """Add gender column to all relevant dataframes"""
    # Men's data: 0
    # Women's data: 1

    data_dict["m_teams"]["Gender"] = 0
    data_dict["w_teams"]["Gender"] = 1

    data_dict["m_seeds"]["Gender"] = 0
    data_dict["w_seeds"]["Gender"] = 1

    data_dict["m_tourney_compact"]["Gender"] = 0
    data_dict["w_tourney_compact"]["Gender"] = 1

    data_dict["m_reg_detailed"]["Gender"] = 0
    data_dict["w_reg_detailed"]["Gender"] = 1

    data_dict["m_tourney_detailed"]["Gender"] = 0
    data_dict["w_tourney_detailed"]["Gender"] = 1

    data_dict["m_conf"]["Gender"] = 0
    data_dict["w_conf"]["Gender"] = 1

    return data_dict

# Load data
data_dict = load_data()
data_dict = add_gender(data_dict)


Loading data files...


In [11]:
def create_team_features(data_dict, season):
    """Create team-level features for a given season"""
    print(f"Creating team features for season {season}...")

    features_list = []

    for gender, gender_label in [(0, "M"), (1, "W")]:
        # Get appropriate datasets
        if gender == 0:
            teams = data_dict["m_teams"].copy()
            reg_detailed = data_dict["m_reg_detailed"]
            tourney_detailed = data_dict["m_tourney_detailed"]
            seeds = data_dict["m_seeds"]
            rankings = data_dict["m_rankings"]
            conf = data_dict["m_conf"]
        else:
            teams = data_dict["w_teams"].copy()
            reg_detailed = data_dict["w_reg_detailed"]
            tourney_detailed = data_dict["w_tourney_detailed"]
            seeds = data_dict["w_seeds"]
            rankings = None  # Women's rankings not available
            conf = data_dict["w_conf"]

        # Filter to teams active in 2026
        if gender == 0:  # Men
            teams = teams[teams["LastD1Season"] >= season].copy()
        teams = teams[["TeamID", "TeamName", "Gender"]].copy()

        # Get current season seeds
        season_seeds = seeds[seeds["Season"] == season].copy()
        season_seeds["SeedNum"] = season_seeds["Seed"].apply(extract_seed_number)
        teams = teams.merge(
            season_seeds[["TeamID", "SeedNum"]], on="TeamID", how="left"
        )

        # Get historical data for features (last 3 seasons)
        hist_start = max(season - 3, 2003 if gender == 0 else 1998)
        hist_reg = reg_detailed[
            (reg_detailed["Season"] >= hist_start) & (reg_detailed["Season"] < season)
        ]
        hist_tourney = tourney_detailed[
            (tourney_detailed["Season"] >= hist_start)
            & (tourney_detailed["Season"] < season)
        ]

        # Calculate team statistics from historical games
        for team_id in teams["TeamID"].unique():
            team_data = {"TeamID": team_id, "Gender": gender}

            # Games as winner and loser
            wins = hist_reg[hist_reg["WTeamID"] == team_id]
            losses = hist_reg[hist_reg["LTeamID"] == team_id]

            # Win percentage
            total_games = len(wins) + len(losses)
            if total_games > 0:
                team_data["WinPct"] = len(wins) / total_games
            else:
                team_data["WinPct"] = 0.5

            # Average points scored and allowed
            if len(wins) > 0:
                avg_pts_scored_w = wins["WScore"].mean()
                avg_pts_allowed_w = wins["LScore"].mean()

                # Detailed stats when winning
                team_data["AvgFGPct_W"] = (
                    wins["WFGM"].sum() / wins["WFGA"].sum()
                    if wins["WFGA"].sum() > 0
                    else 0
                )
                team_data["AvgFG3Pct_W"] = (
                    wins["WFGM3"].sum() / wins["WFGA3"].sum()
                    if wins["WFGA3"].sum() > 0
                    else 0
                )
                team_data["AvgFTPct_W"] = (
                    wins["WFTM"].sum() / wins["WFTA"].sum()
                    if wins["WFTA"].sum() > 0
                    else 0
                )
            else:
                avg_pts_scored_w = 0
                avg_pts_allowed_w = 0
                team_data["AvgFGPct_W"] = 0
                team_data["AvgFG3Pct_W"] = 0
                team_data["AvgFTPct_W"] = 0

            if len(losses) > 0:
                avg_pts_scored_l = losses["LScore"].mean()
                avg_pts_allowed_l = losses["WScore"].mean()

                # Detailed stats when losing
                team_data["AvgFGPct_L"] = (
                    losses["LFGM"].sum() / losses["LFGA"].sum()
                    if losses["LFGA"].sum() > 0
                    else 0
                )
                team_data["AvgFG3Pct_L"] = (
                    losses["LFGM3"].sum() / losses["LFGA3"].sum()
                    if losses["LFGA3"].sum() > 0
                    else 0
                )
                team_data["AvgFTPct_L"] = (
                    losses["LFTM"].sum() / losses["LFTA"].sum()
                    if losses["LFTA"].sum() > 0
                    else 0
                )
            else:
                avg_pts_scored_l = 0
                avg_pts_allowed_l = 0
                team_data["AvgFGPct_L"] = 0
                team_data["AvgFG3Pct_L"] = 0
                team_data["AvgFTPct_L"] = 0

            # Overall averages
            if total_games > 0:
                team_data["AvgPtsScored"] = (
                    avg_pts_scored_w * len(wins) + avg_pts_scored_l * len(losses)
                ) / total_games
                team_data["AvgPtsAllowed"] = (
                    avg_pts_allowed_w * len(wins) + avg_pts_allowed_l * len(losses)
                ) / total_games
            else:
                team_data["AvgPtsScored"] = 70
                team_data["AvgPtsAllowed"] = 70

            # Rebounds, assists, turnovers
            if len(wins) > 0:
                team_data["AvgAst_W"] = (
                    wins["WAst"].mean() if "WAst" in wins.columns else 0
                )
                team_data["AvgTO_W"] = (
                    wins["WTO"].mean() if "WTO" in wins.columns else 0
                )
                team_data["AvgStl_W"] = (
                    wins["WStl"].mean() if "WStl" in wins.columns else 0
                )
                team_data["AvgBlk_W"] = (
                    wins["WBlk"].mean() if "WBlk" in wins.columns else 0
                )
            else:
                team_data["AvgAst_W"] = 0
                team_data["AvgTO_W"] = 0
                team_data["AvgStl_W"] = 0
                team_data["AvgBlk_W"] = 0

            if len(losses) > 0:
                team_data["AvgAst_L"] = (
                    losses["LAst"].mean() if "LAst" in losses.columns else 0
                )
                team_data["AvgTO_L"] = (
                    losses["LTO"].mean() if "LTO" in losses.columns else 0
                )
                team_data["AvgStl_L"] = (
                    losses["LStl"].mean() if "LStl" in losses.columns else 0
                )
                team_data["AvgBlk_L"] = (
                    losses["LBlk"].mean() if "LBlk" in losses.columns else 0
                )
            else:
                team_data["AvgAst_L"] = 0
                team_data["AvgTO_L"] = 0
                team_data["AvgStl_L"] = 0
                team_data["AvgBlk_L"] = 0

            # Tournament experience (games played in previous tournaments)
            hist_tourney_wins = hist_tourney[hist_tourney["WTeamID"] == team_id]
            hist_tourney_losses = hist_tourney[hist_tourney["LTeamID"] == team_id]
            team_data["TourneyExperience"] = len(hist_tourney_wins) + len(
                hist_tourney_losses
            )

            # Tournament win rate
            if team_data["TourneyExperience"] > 0:
                team_data["TourneyWinRate"] = (
                    len(hist_tourney_wins) / team_data["TourneyExperience"]
                )
            else:
                team_data["TourneyWinRate"] = 0

            # Rankings (men only)
            if rankings is not None and gender == 0:
                # Get most recent rankings from previous season
                prev_season_rankings = rankings[rankings["Season"] == season - 1]
                if len(prev_season_rankings) > 0:
                    team_rankings = prev_season_rankings[
                        prev_season_rankings["TeamID"] == team_id
                    ]
                    if len(team_rankings) > 0:
                        # Use median rank across all systems
                        team_data["Ranking"] = team_rankings["OrdinalRank"].median()
                    else:
                        team_data["Ranking"] = 200
                else:
                    team_data["Ranking"] = 200
            else:
                team_data["Ranking"] = 200

            # Conference
            team_conf = conf[
                (conf["TeamID"] == team_id) & (conf["Season"] == season - 1)
            ]
            if len(team_conf) > 0:
                team_data["Conference"] = team_conf["ConfAbbrev"].iloc[0]
            else:
                team_data["Conference"] = "Unknown"

            features_list.append(team_data)

    features_df = pd.DataFrame(features_list)

    # Merge with seed information
    m_seeds = data_dict["m_seeds"][data_dict["m_seeds"]["Season"] == season].copy()
    m_seeds["SeedNum"] = m_seeds["Seed"].apply(extract_seed_number)
    features_df = features_df.merge(
        m_seeds[["TeamID", "SeedNum"]], on="TeamID", how="left"
    )

    w_seeds = data_dict["w_seeds"][data_dict["w_seeds"]["Season"] == season].copy()
    w_seeds["SeedNum"] = w_seeds["Seed"].apply(extract_seed_number)
    features_df = features_df.merge(
        w_seeds[["TeamID", "SeedNum"]], on="TeamID", how="left", suffixes=("", "_w")
    )
    features_df["SeedNum"] = features_df["SeedNum"].fillna(features_df["SeedNum_w"])
    features_df = features_df.drop(columns=["SeedNum_w"], errors="ignore")

    # Fill NaN seeds with median
    features_df["SeedNum"] = features_df["SeedNum"].fillna(8)

    return features_df

def extract_seed_number(seed):
    """Extract numeric seed from seed string (e.g., 'W01' -> 1, 'Z16b' -> 16)"""
    if pd.isna(seed):
        return np.nan
    seed_str = str(seed)
    # Remove region letter and play-in suffix
    num_part = "".join([c for c in seed_str if c.isdigit()])
    if num_part:
        return int(num_part)
    return np.nan

def create_training_data(data_dict):
    """Create training dataset from historical tournament results"""
    print("Creating training data...")

    training_data = []

    # Process men's and women's tournaments
    for gender in [0, 1]:
        if gender == 0:
            tourney_compact = data_dict["m_tourney_compact"]
            min_season = 2003
        else:
            tourney_compact = data_dict["w_tourney_compact"]
            min_season = 1998

        # Use data up to 2025 for training
        for season in range(min_season, 2026):
            print(
                f"  Processing season {season} ({'Men' if gender == 0 else 'Women'})..."
            )

            # Create team features for this season
            team_features = create_team_features(data_dict, season)

            # Get tournament games for this season
            season_games = tourney_compact[tourney_compact["Season"] == season]

            for _, game in season_games.iterrows():
                team1_id = game["WTeamID"]
                team2_id = game["LTeamID"]

                # Create features for winner vs loser (team1 is winner)
                matchup = create_matchup_features(team_features, team1_id, team2_id)
                if matchup is not None:
                    matchup["Target"] = 1  # Team1 (winner) wins
                    matchup["Season"] = season
                    training_data.append(matchup)

                # Also create reverse matchup (loser vs winner) with target 0
                matchup_rev = create_matchup_features(team_features, team2_id, team1_id)
                if matchup_rev is not None:
                    matchup_rev["Target"] = 0  # Team1 (loser) loses
                    matchup_rev["Season"] = season
                    training_data.append(matchup_rev)

    training_df = pd.DataFrame(training_data)
    print(f"Created {len(training_df)} training samples")

    return training_df

def create_matchup_features(team_features, team1_id, team2_id):
    """Create matchup features between two teams"""
    team1 = team_features[team_features["TeamID"] == team1_id]
    team2 = team_features[team_features["TeamID"] == team2_id]

    if len(team1) == 0 or len(team2) == 0:
        return None

    team1 = team1.iloc[0]
    team2 = team2.iloc[0]

    # Ensure same gender
    if team1["Gender"] != team2["Gender"]:
        return None

    features = {}

    # Differences (Team1 - Team2)
    features["WinPct_Diff"] = team1["WinPct"] - team2["WinPct"]
    features["AvgPtsScored_Diff"] = team1["AvgPtsScored"] - team2["AvgPtsScored"]
    features["AvgPtsAllowed_Diff"] = team1["AvgPtsAllowed"] - team2["AvgPtsAllowed"]
    features["SeedNum_Diff"] = team1["SeedNum"] - team2["SeedNum"]
    features["Ranking_Diff"] = team1["Ranking"] - team2["Ranking"]
    features["TourneyExperience_Diff"] = (
        team1["TourneyExperience"] - team2["TourneyExperience"]
    )
    features["TourneyWinRate_Diff"] = team1["TourneyWinRate"] - team2["TourneyWinRate"]

    # Shooting percentages
    features["FGPct_W_Diff"] = team1["AvgFGPct_W"] - team2["AvgFGPct_W"]
    features["FG3Pct_W_Diff"] = team1["AvgFG3Pct_W"] - team2["AvgFG3Pct_W"]
    features["FTPct_W_Diff"] = team1["AvgFTPct_W"] - team2["AvgFTPct_W"]

    # Team 1 stats
    features["Team1_WinPct"] = team1["WinPct"]
    features["Team1_Seed"] = team1["SeedNum"]
    features["Team1_Ranking"] = team1["Ranking"]

    # Team 2 stats
    features["Team2_WinPct"] = team2["WinPct"]
    features["Team2_Seed"] = team2["SeedNum"]
    features["Team2_Ranking"] = team2["Ranking"]

    # Gender
    features["Gender"] = team1["Gender"]

    return features

# Create training data
training_df = create_training_data(data_dict)
training_df

Creating training data...
  Processing season 2003 (Men)...
Creating team features for season 2003...
  Processing season 2004 (Men)...
Creating team features for season 2004...
  Processing season 2005 (Men)...
Creating team features for season 2005...
  Processing season 2006 (Men)...
Creating team features for season 2006...
  Processing season 2007 (Men)...
Creating team features for season 2007...
  Processing season 2008 (Men)...
Creating team features for season 2008...
  Processing season 2009 (Men)...
Creating team features for season 2009...
  Processing season 2010 (Men)...
Creating team features for season 2010...
  Processing season 2011 (Men)...
Creating team features for season 2011...
  Processing season 2012 (Men)...
Creating team features for season 2012...
  Processing season 2013 (Men)...
Creating team features for season 2013...
  Processing season 2014 (Men)...
Creating team features for season 2014...
  Processing season 2015 (Men)...
Creating team features for s

In [ ]:
def train_neural_network(training_df):
    """Train a neural network model"""
    print("Training neural network...")

    # Prepare features
    feature_cols = [
        col for col in training_df.columns if col not in ["Target", "Season"]
    ]

    X = training_df[feature_cols].fillna(0).values
    y = training_df["Target"].values

    print(f"Features: {len(feature_cols)}")
    print(f"Training samples: {len(X)}")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train model
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size="auto",
        learning_rate="adaptive",
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=True,
    )

    model.fit(X_train_scaled, y_train)

    # Evaluate
    train_score = model.score(X_train_scaled, y_train)
    test_score = model.score(X_test_scaled, y_test)
    print(f"\nTraining accuracy: {train_score:.4f}")
    print(f"Test accuracy: {test_score:.4f}")

    return model, scaler, feature_cols


def generate_predictions(data_dict, model, scaler, feature_cols):
    """Generate predictions for 2026 matchups"""
    print("Generating predictions for 2026...")

    # Get team features for 2026
    team_features = create_team_features(data_dict, 2026)

    # Parse sample submission to get matchups
    sample_sub = data_dict["sample_sub"]

    predictions = []

    for idx, row in sample_sub.iterrows():
        if idx % 10000 == 0:
            print(f"  Processed {idx}/{len(sample_sub)} matchups...")

        # Parse ID: SSSS_XXXX_YYYY
        parts = row["ID"].split("_")
        season = int(parts[0])
        team1_id = int(parts[1])
        team2_id = int(parts[2])

        # Create matchup features
        matchup = create_matchup_features(team_features, team1_id, team2_id)

        if matchup is not None:
            # Extract features in correct order
            X = np.array([[matchup.get(col, 0) for col in feature_cols]])
            X_scaled = scaler.transform(X)

            # Predict probability that team1 wins
            prob = model.predict_proba(X_scaled)[0][1]
        else:
            # If we can't create features, use 0.5 (no information)
            prob = 0.5

        predictions.append({"ID": row["ID"], "Pred": prob})

    predictions_df = pd.DataFrame(predictions)
    print(f"Generated {len(predictions_df)} predictions")

    return predictions_df

# Train model
model, scaler, feature_cols = train_neural_network(training_df)

# Generate predictions
predictions_df = generate_predictions(data_dict, model, scaler, feature_cols)

Training neural network...
Features: 17
Training samples: 6332
Iteration 1, loss = 0.64016844
Validation score: 0.741617
Iteration 2, loss = 0.54412502
Validation score: 0.741617
Iteration 3, loss = 0.51453191
Validation score: 0.753452
Iteration 4, loss = 0.50744325
Validation score: 0.743590
Iteration 5, loss = 0.50209763
Validation score: 0.755424
Iteration 6, loss = 0.49890311
Validation score: 0.737673
Iteration 7, loss = 0.49745320
Validation score: 0.739645
Iteration 8, loss = 0.49476067
Validation score: 0.735700
Iteration 9, loss = 0.49289430
Validation score: 0.745562
Iteration 10, loss = 0.49050565
Validation score: 0.747535
Iteration 11, loss = 0.48845482
Validation score: 0.735700
Iteration 12, loss = 0.48731659
Validation score: 0.737673
Iteration 13, loss = 0.48575501
Validation score: 0.747535
Iteration 14, loss = 0.48516074
Validation score: 0.739645
Iteration 15, loss = 0.48258768
Validation score: 0.737673
Iteration 16, loss = 0.48182375
Validation score: 0.743590
It

In [ ]:
# Save predictions
output_file = "submission.csv"
predictions_df.to_csv(output_file, index=False)
print(f"\nPredictions saved to {output_file}")
print(f"Total predictions: {len(predictions_df)}")
print(f"\nSample predictions:")
print(predictions_df.head(10))